# Lagrangian Particle Flows for the Fokker-Planck Equation

## Overview

The **Fokker-Planck equation** describes the time evolution of the probability density $\rho(x, t)$ of a stochastic process driven by drift and diffusion. It appears across physics (Brownian motion, statistical mechanics), machine learning (diffusion models, Langevin MCMC), and mathematical biology (population dynamics).

Rather than solving for the density $\rho$ on a grid (the Eulerian approach), the **Lagrangian particle method** tracks individual samples $\{x_i(t)\}$ whose empirical distribution approximates $\rho(t, \cdot)$. This perspective connects the PDE to stochastic differential equations and gradient flows.

## The Fokker-Planck PDE

The Fokker-Planck equation in $\mathbb{R}^d$:
$$
\partial_t \rho = \nabla \cdot (\rho\, \nabla V) + \kappa\, \Delta \rho
$$
- $V(x)$: external potential (drift term attracts particles to minima of $V$)
- $\kappa > 0$: diffusion coefficient (controls exploration/noise)
- Stationary distribution: $\rho_\infty \propto e^{-V(x)/\kappa}$ (Gibbs/Boltzmann measure)

## Lagrangian Discretization

We discretize $\rho(t)$ by $N$ particles $\{x_i(t)\}_{i=1}^N \approx \rho(t, \cdot)$.

**Stochastic (Langevin) dynamics** — the particle-level SDE corresponding to the Fokker-Planck PDE:
$$
dx_i = -\nabla V(x_i)\, dt + \sqrt{2\kappa}\, dW_i
$$
where $W_i$ are independent standard Brownian motions.

**Deterministic particle flow** — instead of stochastic noise, particles repel each other to approximate the entropic pressure $\kappa \nabla \log \rho$. In 1D, interaction-based approximation:
$$
\dot{x}_i = -\nabla V(x_i) + \kappa \sum_{j \neq i} \frac{1}{x_i - x_j}
$$
The repulsive term $\sum_{j\neq i}(x_i - x_j)^{-1}$ approximates $\nabla \log \rho$ by gradient of log-distances.

## Double-Well Potential

We study the double-well potential in 2D:
$$
V(x, y) = \frac{(x-y)^2}{2} + 0.5(x^2 - 1)^2
$$
This potential has two minima at approximately $(\pm 1, \pm 1)$, with an energy barrier between them. The parameter $\kappa$ controls how much the noise allows transitions across the barrier.

## What This Notebook Demonstrates

1. Langevin dynamics: particles starting near origin spreading toward the double-well equilibrium
2. Effect of noise level $\kappa$: low noise traps particles; high noise explores both wells
3. Comparison of particle distributions at different times
4. Stationary distribution $\rho_\infty \propto e^{-V/\kappa}$ as a reference

### Setup

We use numpy for the Euler-Maruyama integration of the Langevin SDE. The time step $\Delta t$ must be small enough for numerical stability; we use $\Delta t = 0.01$. The potential $V$ and its gradient $\nabla V$ are implemented analytically.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})

def V(x, y):
    """Double-well potential V(x,y) = (x-y)^2/2 + 0.5*(x^2-1)^2."""
    return 0.5 * (x - y)**2 + 0.5 * (x**2 - 1)**2

def gradV(xy):
    """Gradient of V at array xy of shape (N, 2)."""
    x, y = xy[:, 0], xy[:, 1]
    dVdx = (x - y) + 2.0 * x * (x**2 - 1)
    dVdy = -(x - y)
    return np.stack([dVdx, dVdy], axis=1)

print("Potential V and gradient defined.")
print(f"V(1,1) = {V(1,1):.4f} (minimum)")
print(f"V(-1,-1) = {V(-1,-1):.4f} (minimum)")
print(f"V(0,0) = {V(0,0):.4f} (saddle point)")

Potential V and gradient defined.
V(1,1) = 0.0000 (minimum)
V(-1,-1) = 0.0000 (minimum)
V(0,0) = 0.5000 (saddle point)


### Potential Landscape Visualization

Before running particles, we visualize the double-well potential landscape. The energy wells are at $(\pm 1, \pm 1)$ and the barrier runs diagonally. The Boltzmann distribution $\rho_\infty(x,y) \propto e^{-V(x,y)/\kappa}$ concentrates mass in the two wells, with the balance controlled by $\kappa$.

For small $\kappa$, the distribution is tightly concentrated near the minima. For large $\kappa$, the distribution spreads and the two wells become well-connected.

In [2]:
xg = np.linspace(-2.5, 2.5, 200)
yg = np.linspace(-2.5, 2.5, 200)
XX, YY = np.meshgrid(xg, yg)
VV = V(XX, YY)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

kappas_pot = [0.1, 0.5, 2.0]
for ax, kap in zip(axes, kappas_pot):
    gibbs = np.exp(-VV / kap)
    gibbs /= gibbs.sum()
    ax.contourf(XX, YY, gibbs, levels=30, cmap='hot_r')
    ax.contour(XX, YY, VV, levels=10, colors='white', alpha=0.4, linewidths=0.8)
    ax.set_title(f'$\\rho_\\infty \\propto e^{{-V/\\kappa}}$, $\\kappa={kap}$')
    ax.set_xlabel('$x$')
    ax.set_ylabel('$y$')
    ax.set_aspect('equal')
    ax.plot([1, -1], [1, -1], 'w+', ms=12, mew=2)  # mark minima

plt.suptitle('Double-Well Potential: Gibbs Distribution for Various $\\kappa$',
             fontsize=13)
plt.tight_layout()
plt.savefig("potential_landscape.png", dpi=80, bbox_inches='tight')
plt.show()

/var/folders/c3/8qf_y_jj6393y3l0dl0bb3k80000gp/T/ipykernel_51622/1925646558.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Euler-Maruyama Langevin Dynamics

We integrate the Langevin SDE:
$$
dx_i = -\nabla V(x_i)\, dt + \sqrt{2\kappa}\, dW_i
$$
using the **Euler-Maruyama** scheme:
$$
x_i^{k+1} = x_i^k - \nabla V(x_i^k)\, \Delta t + \sqrt{2\kappa \Delta t}\, \xi_i^k
$$
where $\xi_i^k \sim \mathcal{N}(0, I_2)$ are i.i.d. standard Gaussian increments.

We run $N = 300$ particles initialized near the origin for $T = 200$ steps with $\Delta t = 0.05$.

In [3]:
def run_langevin(N, kappa, n_steps, dt, seed=42):
    """
    Run Langevin dynamics for N particles.
    Returns trajectory: shape (n_steps+1, N, 2)
    """
    rng = np.random.default_rng(seed)
    # Initialize near origin with small spread
    X = rng.standard_normal((N, 2)) * 0.3
    traj = [X.copy()]
    noise_scale = np.sqrt(2 * kappa * dt)

    for _ in range(n_steps):
        grad = gradV(X)
        noise = rng.standard_normal((N, 2))
        X = X - grad * dt + noise_scale * noise
        traj.append(X.copy())

    return np.array(traj)


N_particles = 300
dt = 0.05
n_steps = 200

kappas = [0.1, 0.5, 1.5]
trajectories = {}
for kap in kappas:
    trajectories[kap] = run_langevin(N_particles, kap, n_steps, dt)

print(f"Trajectories computed: {n_steps} steps, {N_particles} particles.")
print(f"Trajectory shape: {trajectories[kappas[0]].shape}")

Trajectories computed: 200 steps, 300 particles.
Trajectory shape: (201, 300, 2)


### Temporal Evolution of Particle Distributions

We display snapshots of the particle cloud at times $t = 0, 2, 5, 10$ (in units of $\Delta t = 0.05$, i.e., steps 0, 40, 100, 200) for $\kappa = 0.5$. Early on, the particles are concentrated near the origin; they gradually spread toward the two energy minima, with dynamics driven by the gradient of $V$ and diffusion from the Brownian noise.

In [4]:
kap_show = 0.5
traj = trajectories[kap_show]
snapshot_steps = [0, 40, 100, 200]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Background: Gibbs distribution
gibbs_bg = np.exp(-VV / kap_show)
gibbs_bg /= gibbs_bg.max()

for ax, step in zip(axes, snapshot_steps):
    X_snap = traj[step]
    ax.contourf(XX, YY, gibbs_bg, levels=20, cmap='Blues', alpha=0.4)
    ax.scatter(X_snap[:, 0], X_snap[:, 1], s=8, alpha=0.6,
               color='crimson', zorder=3)
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(-2.5, 2.5)
    ax.set_aspect('equal')
    ax.set_title(f'$t = {step * dt:.1f}$')
    ax.set_xlabel('$x$')
    ax.set_ylabel('$y$')
    ax.plot([1, -1], [1, -1], 'b+', ms=10, mew=2, label='Minima')

plt.suptitle(f'Langevin Dynamics, $\\kappa={kap_show}$: Particles Evolve Toward Equilibrium',
             fontsize=13)
plt.tight_layout()
plt.savefig("langevin_evolution.png", dpi=80, bbox_inches='tight')
plt.show()

/var/folders/c3/8qf_y_jj6393y3l0dl0bb3k80000gp/T/ipykernel_51622/857301507.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Effect of Noise Level $\kappa$

The competition between drift and diffusion is controlled by $\kappa$:
- **Small $\kappa$**: drift dominates, particles quickly settle in the nearest well and stay there (metastability)
- **Large $\kappa$**: diffusion dominates, particles explore both wells freely

We compare the final particle distributions ($t = n_{\rm steps} \cdot \Delta t$) for three values of $\kappa$ against the corresponding Gibbs distribution $e^{-V/\kappa}$.

In [5]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for col, kap in enumerate(kappas):
    traj = trajectories[kap]
    X_final = traj[-1]
    gibbs = np.exp(-VV / kap)
    gibbs /= gibbs.max()

    # Top row: particle scatter
    ax_top = axes[0, col]
    ax_top.contourf(XX, YY, gibbs, levels=20, cmap='Blues', alpha=0.5)
    ax_top.scatter(X_final[:, 0], X_final[:, 1], s=10, alpha=0.7,
                   color='crimson', zorder=3)
    ax_top.plot([1, -1], [1, -1], 'b+', ms=12, mew=2)
    ax_top.set_xlim(-2.5, 2.5)
    ax_top.set_ylim(-2.5, 2.5)
    ax_top.set_aspect('equal')
    ax_top.set_title(f'$\\kappa = {kap}$ — Final particles')

    # Bottom row: KDE histogram
    ax_bot = axes[1, col]
    h, xe, ye = np.histogram2d(X_final[:, 0], X_final[:, 1],
                                bins=30, range=[[-2.5, 2.5], [-2.5, 2.5]],
                                density=True)
    ax_bot.imshow(h.T, origin='lower', extent=[-2.5, 2.5, -2.5, 2.5],
                  cmap='hot_r', aspect='equal')
    ax_bot.contour(XX, YY, gibbs, levels=6, colors='cyan', alpha=0.6, linewidths=1)
    ax_bot.set_title(f'$\\kappa = {kap}$ — Empirical density')
    ax_bot.set_xlabel('$x$')
    ax_bot.set_ylabel('$y$')

plt.suptitle('Langevin Dynamics: Effect of Noise Level $\\kappa$', fontsize=14)
plt.tight_layout()
plt.savefig("kappa_comparison.png", dpi=80, bbox_inches='tight')
plt.show()

/var/folders/c3/8qf_y_jj6393y3l0dl0bb3k80000gp/T/ipykernel_51622/3673003134.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Interactive Animation

The widget below animates the Langevin particle dynamics with a play/stop control and time slider. Choose the noise level $\kappa$ from the dropdown. The blue heatmap shows the target Gibbs distribution $e^{-V/\kappa}$; red dots are the current particle positions.

### Static Snapshot

This cell generates the snippet: a side-by-side panel showing the Langevin particle clouds at the beginning and end of the simulation for the three $\kappa$ values, with the Gibbs reference overlaid.

In [6]:
STATIC_SNAPSHOT = True

if STATIC_SNAPSHOT:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    fig_s, axes_s = plt.subplots(1, 3, figsize=(12, 4))
    fig_s.patch.set_facecolor('#111111')

    for ax, kap in zip(axes_s, kappas):
        gibbs_s = np.exp(-VV / kap)
        gibbs_s /= gibbs_s.max()
        X_fin = trajectories[kap][-1]
        ax.set_facecolor('#111111')
        ax.contourf(XX, YY, gibbs_s, levels=20, cmap='Blues', alpha=0.6)
        ax.scatter(X_fin[:, 0], X_fin[:, 1], s=10, alpha=0.8,
                   color='tomato', zorder=3)
        ax.plot([1, -1], [1, -1], 'w+', ms=12, mew=2)
        ax.set_xlim(-2.5, 2.5)
        ax.set_ylim(-2.5, 2.5)
        ax.set_aspect('equal')
        ax.set_title(f'$\\kappa={kap}$', color='white', fontsize=13)
        ax.tick_params(colors='white')
        for spine in ax.spines.values():
            spine.set_edgecolor('white')

    plt.suptitle('Lagrangian Flows — Langevin at Equilibrium',
                 color='white', fontsize=13)
    plt.tight_layout()
    plt.savefig("snippet.png", dpi=100, bbox_inches='tight',
                facecolor=fig_s.get_facecolor())
    plt.close()
    print("snippet.png saved.")

snippet.png saved.


## Takeaways

- The **Fokker-Planck equation** $\partial_t \rho = \nabla \cdot (\rho \nabla V) + \kappa \Delta \rho$ governs the evolution of probability densities under drift-diffusion dynamics.
- The **Lagrangian particle method** tracks sample trajectories $x_i(t)$ whose empirical measure approximates $\rho(t)$; it transforms the PDE into an ensemble of SDEs.
- **Euler-Maruyama** discretizes the Langevin SDE $dx = -\nabla V\, dt + \sqrt{2\kappa}\, dW$ with a simple explicit step, producing correct long-time statistics (ergodicity).
- The **noise level $\kappa$** controls a fundamental trade-off: low $\kappa$ gives sharp concentration near energy minima (metastability), high $\kappa$ enables exploration of the full landscape.
- The **stationary distribution** is the Gibbs measure $\rho_\infty \propto e^{-V/\kappa}$; the particle ensemble converges to this target in distribution.

## Bibliography

- **Risken, H.** (1996). *The Fokker-Planck Equation: Methods of Solution and Applications.* Springer.
- **Welling, M. & Teh, Y. W.** (2011). *Bayesian learning via stochastic gradient Langevin dynamics.* ICML.
- **Jordan, R., Kinderlehrer, D. & Otto, F.** (1998). *The variational formulation of the Fokker-Planck equation.* SIAM Journal on Mathematical Analysis, 29(1), 1–17.
- **Liu, Q. & Wang, D.** (2016). *Stein variational gradient descent: A general purpose Bayesian inference algorithm.* NeurIPS.
- **Pavliotis, G. A.** (2014). *Stochastic Processes and Applications.* Springer Texts in Applied Mathematics.